# Benchmark Model Throughput - Colab Self-Contained

Notebook này không phụ thuộc vào `scripts/benchmark_model_throughput.py`. Nó dùng trực tiếp đường dẫn giống các notebook có sẵn: `prepared_data_v1/...` và `Model/...`.

Mặc định benchmark toàn bộ test set giống notebook 06 cho `YOLOv11m + EfficientNetB2` và các baseline phụ `DenseNet121/ResNet50/VGG16`.

In [ ]:
# CELL 1: cài thư viện
%pip -q install ultralytics pandas==2.2.2 pillow==11.3.0 tqdm gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.3 MB/s eta 0:00:00


In [ ]:
# CELL 2: mount Drive và tìm project root giống notebook 06
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import os

PROJECT_NAME = "Papaya_Leaf_Disease_Classification"
MANUAL_PROJECT_ROOT = None
# MANUAL_PROJECT_ROOT = Path("/content/drive/MyDrive/Papaya_Leaf_Disease_Classification")

def find_project_root():
    if MANUAL_PROJECT_ROOT is not None:
        p = Path(MANUAL_PROJECT_ROOT)
        if p.exists():
            return p
        raise FileNotFoundError(f"MANUAL_PROJECT_ROOT không tồn tại: {p}")
    p = Path("/content/drive/MyDrive") / PROJECT_NAME
    if p.exists():
        return p
    for root in [Path("/content/drive/MyDrive"), Path("/content/drive/.shortcut-targets-by-id")]:
        if root.exists():
            for found in root.rglob(PROJECT_NAME):
                if found.is_dir() and (found / "prepared_data_v1").exists():
                    return found
    raise FileNotFoundError("Không tìm thấy project root. Hãy set MANUAL_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

PREP_ROOT = PROJECT_ROOT / "prepared_data_v1"
YOLO_IMAGES_TEST = PREP_ROOT / "yolo_dataset" / "images" / "test"
YOLO_LABELS_TEST = PREP_ROOT / "yolo_dataset" / "labels" / "test"
CNN_HEALTHY_TEST = PREP_ROOT / "cnn_dataset" / "test" / "Healthy"
MODEL_ROOT = PROJECT_ROOT / "Model"
YOLO_MODEL_ROOT = MODEL_ROOT / "YOLOv11"
CNN_MODEL_ROOT = MODEL_ROOT / "CNN"
CNN_BASELINE_ROOT = MODEL_ROOT / "CNN_Baselines"
YOLO_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "yolo"
CNN_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "cnn"
CNN_BASELINE_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "cnn_baselines"
BENCHMARK_ROOT = MODEL_ROOT / "Performance_Benchmark"
BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)

for name, path in {
    "PROJECT_ROOT": PROJECT_ROOT,
    "YOLO_IMAGES_TEST": YOLO_IMAGES_TEST,
    "YOLO_LABELS_TEST": YOLO_LABELS_TEST,
    "CNN_HEALTHY_TEST": CNN_HEALTHY_TEST,
    "YOLO_MODEL_ROOT": YOLO_MODEL_ROOT,
    "CNN_MODEL_ROOT": CNN_MODEL_ROOT,
    "CNN_BASELINE_ROOT": CNN_BASELINE_ROOT,
    "BENCHMARK_ROOT": BENCHMARK_ROOT,
}.items():
    print(f"{name:18}: {path} | exists={path.exists()}")

assert YOLO_IMAGES_TEST.exists(), YOLO_IMAGES_TEST
assert YOLO_LABELS_TEST.exists(), YOLO_LABELS_TEST
assert CNN_HEALTHY_TEST.exists(), CNN_HEALTHY_TEST
assert YOLO_MODEL_ROOT.exists() or YOLO_OUTPUT_ROOT.exists(), f"No YOLO model folder under {PROJECT_ROOT}/Model or outputs"
assert CNN_MODEL_ROOT.exists() or CNN_OUTPUT_ROOT.exists(), f"No CNN model folder under {PROJECT_ROOT}/Model or outputs"
assert CNN_BASELINE_ROOT.exists() or CNN_BASELINE_OUTPUT_ROOT.exists(), f"No CNN baseline folder under {PROJECT_ROOT}/Model or outputs"

In [ ]:
# CELL 3: cấu hình benchmark
import gc, json, random, statistics, time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from PIL import Image
from tensorflow import keras
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
YOLO_CLASS_NAMES = ["Anthracnose", "BacterialSpot", "Curl", "RingSpot"]
CNN_CLASS_NAMES = ["Anthracnose", "BacterialSpot", "Curl", "Healthy", "RingSpot"]
ALL_CLASS_NAMES = CNN_CLASS_NAMES
YOLO_ID_TO_CLASS = {i: name for i, name in enumerate(YOLO_CLASS_NAMES)}
CNN_ID_TO_CLASS = {i: name for i, name in enumerate(CNN_CLASS_NAMES)}

CNN_SPECS = {
    "EfficientNetB0": {"img_size": 224, "preprocess_fn": None, "root": CNN_MODEL_ROOT, "dirs": ["EfficientNetB0", "efficientnet_b0"]},
    "EfficientNetB1": {"img_size": 240, "preprocess_fn": None, "root": CNN_MODEL_ROOT, "dirs": ["EfficientNetB1", "efficientnet_b1"]},
    "EfficientNetB2": {"img_size": 260, "preprocess_fn": None, "root": CNN_MODEL_ROOT, "dirs": ["EfficientNetB2", "efficientnet_b2"]},
    "DenseNet121": {"img_size": 224, "preprocess_fn": tf.keras.applications.densenet.preprocess_input, "root": CNN_BASELINE_ROOT, "dirs": ["DenseNet121", "densenet_121"]},
    "ResNet50": {"img_size": 224, "preprocess_fn": tf.keras.applications.resnet50.preprocess_input, "root": CNN_BASELINE_ROOT, "dirs": ["ResNet50", "resnet_50"]},
    "VGG16": {"img_size": 224, "preprocess_fn": tf.keras.applications.vgg16.preprocess_input, "root": CNN_BASELINE_ROOT, "dirs": ["VGG16", "vgg_16"]},
}

YOLO_ALIAS_DIRS = {
    "YOLOv11n": ["YOLOv11n", "yolov11_n"],
    "YOLOv11s": ["YOLOv11s", "yolov11_s"],
    "YOLOv11m": ["YOLOv11m", "yolov11_m"],
}

PRESETS = {
    "main-cascade": (["YOLOv11m"], ["EfficientNetB2"]),
    "baseline-cascades": (["YOLOv11m"], ["DenseNet121", "ResNet50", "VGG16"]),
    "paper-cascades": (["YOLOv11m"], ["EfficientNetB2", "DenseNet121", "ResNet50", "VGG16"]),
    "notebook06-efficientnets": (["YOLOv11n", "YOLOv11s", "YOLOv11m"], ["EfficientNetB0", "EfficientNetB1", "EfficientNetB2"]),
}

PRESET = "paper-cascades"
LIMIT = 0
WARMUP = 3
BATCH_CROPS = False

YOLO_IMGSZ = 832
YOLO_IOU = 0.70
YOLO_CONF_START = 0.50
YOLO_CONF_MIN = 0.35
YOLO_CONF_STEP = 0.05
FULL_CNN_DISEASE_MIN_PROB = 0.50
BBOX_EXPAND_RATIO = 0.10
MIN_CROP_SIZE = 8
AGG_METHOD = "weighted"
TORCH_DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("PRESET              :", PRESET)
print("Torch CUDA available:", torch.cuda.is_available())
print("TORCH_DEVICE        :", TORCH_DEVICE)
print("TensorFlow version  :", tf.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PRESET              : paper-cascades
Torch CUDA available: True
TORCH_DEVICE        : 0
TensorFlow version  : 2.20.0


In [ ]:
# Find checkpoints in either Model/ or outputs/.
def normalized(path):
    return str(path).lower().replace('_', '').replace('-', '').replace(' ', '')

def find_yolo_checkpoint(alias):
    alias_dirs = YOLO_ALIAS_DIRS.get(alias, [alias])
    for alias_dir in alias_dirs:
        for root in [YOLO_MODEL_ROOT, YOLO_OUTPUT_ROOT]:
            for name in ['best.pt', 'last.pt']:
                candidate = root / alias_dir / 'weights' / name
                if candidate.is_file():
                    return candidate
    key = normalized(alias)
    found = [p for p in PROJECT_ROOT.rglob('*.pt') if key in normalized(p) and p.name in {'best.pt', 'last.pt'}]
    for name in ['best.pt', 'last.pt']:
        matches = sorted(p for p in found if p.name == name)
        if matches:
            return matches[0]
    return None

def find_cnn_checkpoint(alias):
    spec = CNN_SPECS[alias]
    priority = ['best_model.keras', 'best_stage2.keras', 'last_stage2.keras', 'best_stage1.keras', 'last_stage1.keras']
    if spec['root'] == CNN_MODEL_ROOT:
        roots = [CNN_MODEL_ROOT, CNN_OUTPUT_ROOT]
    else:
        roots = [CNN_BASELINE_ROOT, CNN_BASELINE_OUTPUT_ROOT]
    for root in roots:
        for alias_dir in spec['dirs']:
            for name in priority:
                candidate = root / alias_dir / name
                if candidate.is_file():
                    return candidate
    key_options = [normalized(alias_dir) for alias_dir in spec['dirs']]
    found = [p for p in PROJECT_ROOT.rglob('*.keras') if any(key in normalized(p) for key in key_options) and p.name in priority]
    if spec['root'] == CNN_BASELINE_ROOT:
        found = [p for p in found if 'cnnbaselines' in normalized(p)]
    else:
        found = [p for p in found if 'cnnbaselines' not in normalized(p)]
    for name in priority:
        matches = sorted(p for p in found if p.name == name)
        if matches:
            return matches[0]
    return None

selected_yolo_aliases, selected_cnn_aliases = PRESETS[PRESET]
available_yolo = {alias: find_yolo_checkpoint(alias) for alias in selected_yolo_aliases}
available_cnn = {alias: find_cnn_checkpoint(alias) for alias in selected_cnn_aliases}
available_yolo = {alias: path for alias, path in available_yolo.items() if path is not None}
available_cnn = {alias: path for alias, path in available_cnn.items() if path is not None}

print('===== AVAILABLE YOLO CHECKPOINTS =====')
for alias, path in available_yolo.items(): print(alias, '=>', path)
print('\n===== AVAILABLE CNN CHECKPOINTS =====')
for alias, path in available_cnn.items(): print(alias, '=>', path)
missing_yolo = [alias for alias in selected_yolo_aliases if alias not in available_yolo]
missing_cnn = [alias for alias in selected_cnn_aliases if alias not in available_cnn]
if missing_yolo: print('\n[WARNING] Missing YOLO checkpoints:', missing_yolo)
if missing_cnn: print('[WARNING] Missing CNN checkpoints:', missing_cnn)
assert available_yolo, f'No YOLO checkpoints found under {PROJECT_ROOT}/Model or outputs.'
assert available_cnn, f'No CNN checkpoints found under {PROJECT_ROOT}/Model or outputs.'


In [ ]:
# CELL 5: tạo test set giống notebook 06
def list_images(folder):
    return sorted([p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTS])

def parse_yolo_label_file(label_path):
    ids = []
    if not Path(label_path).exists():
        return ids
    with open(label_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5: continue
            try:
                cls_id = int(float(parts[0]))
                if cls_id in YOLO_ID_TO_CLASS: ids.append(cls_id)
            except Exception:
                pass
    return ids

def infer_true_class_from_yolo_label(label_path):
    ids = parse_yolo_label_file(label_path)
    if not ids: return None
    return YOLO_ID_TO_CLASS[Counter(ids).most_common(1)[0][0]]

records = []
for img_path in list_images(YOLO_IMAGES_TEST):
    lbl_path = YOLO_LABELS_TEST / f"{img_path.stem}.txt"
    true_class = infer_true_class_from_yolo_label(lbl_path)
    if true_class is None: continue
    records.append({"image_path": str(img_path), "true_class": true_class, "source": "yolo_test_disease", "label_path": str(lbl_path)})
for img_path in list_images(CNN_HEALTHY_TEST):
    records.append({"image_path": str(img_path), "true_class": "Healthy", "source": "cnn_test_healthy", "label_path": None})

test_df = pd.DataFrame(records).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
if LIMIT not in (None, 0):
    test_df = test_df.head(int(LIMIT)).copy()

manifest_path = BENCHMARK_ROOT / "benchmark_test_manifest_notebook06.csv"
test_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")
print("Total test images:", len(test_df))
print("Saved manifest   :", manifest_path)
display(test_df.groupby(["source", "true_class"]).size().reset_index(name="count"))

Total test images: 291
Saved manifest   : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/benchmark_test_manifest_notebook06.csv


,source,true_class,count
0,cnn_test_healthy,Healthy,34
1,yolo_test_disease,Anthracnose,54
2,yolo_test_disease,BacterialSpot,69
3,yolo_test_disease,Curl,54
4,yolo_test_disease,RingSpot,80


In [ ]:
# CELL 6: hàm cascade inference và timing
def sync_torch():
    if TORCH_DEVICE != "cpu" and torch.cuda.is_available():
        torch.cuda.synchronize()

def preprocess_array(alias, arr):
    fn = CNN_SPECS[alias]["preprocess_fn"]
    return arr if fn is None else fn(arr)

def cnn_predict_pil(alias, model, pil_img):
    size = CNN_SPECS[alias]["img_size"]
    arr = np.asarray(pil_img.resize((size, size))).astype("float32")
    arr = preprocess_array(alias, arr)
    probs = model.predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    pred_id = int(np.argmax(probs))
    return CNN_ID_TO_CLASS[pred_id], float(probs[pred_id])

def cnn_predict_batch(alias, model, crops):
    size = CNN_SPECS[alias]["img_size"]
    arrs = []
    for img in crops:
        arr = np.asarray(img.resize((size, size))).astype("float32")
        arrs.append(preprocess_array(alias, arr))
    probs = model.predict(np.stack(arrs, axis=0), verbose=0)
    ids = np.argmax(probs, axis=1)
    return [(CNN_ID_TO_CLASS[int(i)], float(probs[j][int(i)])) for j, i in enumerate(ids)]

def yolo_detect_once(model, image_path, conf):
    results = model.predict(source=str(image_path), imgsz=YOLO_IMGSZ, conf=conf, iou=YOLO_IOU, device=TORCH_DEVICE, verbose=False)
    r = results[0]
    if r.boxes is None or len(r.boxes) == 0: return []
    boxes = []
    for b in r.boxes:
        cls_id = int(b.cls.detach().cpu().numpy()[0])
        if cls_id not in YOLO_ID_TO_CLASS: continue
        boxes.append({"xyxy": b.xyxy.detach().cpu().numpy()[0].astype(float).tolist(), "yolo_class": YOLO_ID_TO_CLASS[cls_id], "yolo_conf": float(b.conf.detach().cpu().numpy()[0])})
    return sorted(boxes, key=lambda x: x["yolo_conf"], reverse=True)

def conf_sequence(start, min_conf, step):
    vals, c = [], start
    while c >= min_conf - 1e-9:
        vals.append(round(c, 3)); c -= step
    return vals

def clamp_box(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(round(x1)), w - 1)); y1 = max(0, min(int(round(y1)), h - 1))
    x2 = max(1, min(int(round(x2)), w)); y2 = max(1, min(int(round(y2)), h))
    if x2 <= x1: x2 = min(w, x1 + 1)
    if y2 <= y1: y2 = min(h, y1 + 1)
    return x1, y1, x2, y2

def crop_from_box(img, box):
    w, h = img.size
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    x1, y1, x2, y2 = clamp_box(x1 - bw * BBOX_EXPAND_RATIO, y1 - bh * BBOX_EXPAND_RATIO, x2 + bw * BBOX_EXPAND_RATIO, y2 + bh * BBOX_EXPAND_RATIO, w, h)
    if (x2 - x1) < MIN_CROP_SIZE or (y2 - y1) < MIN_CROP_SIZE: return None
    return img.crop((x1, y1, x2, y2))

def aggregate(crop_preds):
    if not crop_preds: return "Healthy"
    scores = {c: 0.0 for c in ALL_CLASS_NAMES}
    if AGG_METHOD == "majority":
        votes = Counter([p["cnn_class"] for p in crop_preds])
        best_vote = max(votes.values())
        candidates = [c for c, v in votes.items() if v == best_vote]
        weights = defaultdict(float)
        for p in crop_preds: weights[p["cnn_class"]] += p["cnn_conf"] * p["yolo_conf"]
        return sorted(candidates, key=lambda c: weights[c], reverse=True)[0]
    for p in crop_preds: scores[p["cnn_class"]] += p["cnn_conf"] * p["yolo_conf"]
    return max(scores, key=scores.get)

def run_hybrid(image_path, yolo_model, cnn_alias, cnn_model):
    img = Image.open(image_path).convert("RGB")
    boxes = yolo_detect_once(yolo_model, image_path, YOLO_CONF_START)
    conf_used, retries = YOLO_CONF_START, 0
    if not boxes:
        full_cls, full_conf = cnn_predict_pil(cnn_alias, cnn_model, img)
        if full_cls == "Healthy" or full_conf < FULL_CNN_DISEASE_MIN_PROB:
            return {"pred_class": "Healthy", "reason": "no_yolo_box_full_cnn_healthy_or_low_conf", "num_boxes": 0, "num_valid_crops": 0, "conf_used": conf_used, "conf_retries": retries}
        for c in conf_sequence(YOLO_CONF_START - YOLO_CONF_STEP, YOLO_CONF_MIN, YOLO_CONF_STEP):
            retries += 1
            boxes = yolo_detect_once(yolo_model, image_path, c)
            if boxes:
                conf_used = c; break
        if not boxes:
            return {"pred_class": "Healthy", "reason": "no_yolo_box_after_conf_retry", "num_boxes": 0, "num_valid_crops": 0, "conf_used": YOLO_CONF_MIN, "conf_retries": retries}
    crops, metas, preds = [], [], []
    for box in boxes:
        crop = crop_from_box(img, box["xyxy"])
        if crop is None: continue
        if BATCH_CROPS:
            crops.append(crop); metas.append(box)
        else:
            cls, conf = cnn_predict_pil(cnn_alias, cnn_model, crop)
            preds.append({"cnn_class": cls, "cnn_conf": conf, "yolo_conf": box["yolo_conf"]})
    if BATCH_CROPS and crops:
        for box, (cls, conf) in zip(metas, cnn_predict_batch(cnn_alias, cnn_model, crops)):
            preds.append({"cnn_class": cls, "cnn_conf": conf, "yolo_conf": box["yolo_conf"]})
    if not preds:
        return {"pred_class": "Healthy", "reason": "boxes_exist_but_all_crops_invalid", "num_boxes": len(boxes), "num_valid_crops": 0, "conf_used": conf_used, "conf_retries": retries}
    return {"pred_class": aggregate(preds), "reason": f"yolo_boxes_cnn_{AGG_METHOD}_vote", "num_boxes": len(boxes), "num_valid_crops": len(preds), "conf_used": conf_used, "conf_retries": retries}

def benchmark_pair(yolo_alias, yolo_ckpt, cnn_alias, cnn_ckpt):
    pair = f"{yolo_alias}__{cnn_alias}"
    print("\n" + "="*100); print("BENCHMARK:", pair); print("YOLO:", yolo_ckpt); print("CNN :", cnn_ckpt)
    t0 = time.perf_counter()
    yolo_model = YOLO(str(yolo_ckpt))
    cnn_model = keras.models.load_model(str(cnn_ckpt), compile=False)
    load_s = time.perf_counter() - t0
    for _, r in test_df.head(min(WARMUP, len(test_df))).iterrows():
        run_hybrid(r["image_path"], yolo_model, cnn_alias, cnn_model)
    sync_torch()
    rows, times = [], []
    t0 = time.perf_counter()
    for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc=pair):
        s = time.perf_counter()
        pred = run_hybrid(r["image_path"], yolo_model, cnn_alias, cnn_model)
        sync_torch(); dt = time.perf_counter() - s
        times.append(dt)
        rows.append({"pair_name": pair, "image_path": r["image_path"], "true_class": r["true_class"], "source": r["source"], **pred})
    elapsed = time.perf_counter() - t0
    ms = [x*1000 for x in times]
    correct = sum(1 for row in rows if row["pred_class"] == row["true_class"])
    pair_dir = BENCHMARK_ROOT / pair; pair_dir.mkdir(parents=True, exist_ok=True)
    pred_csv = pair_dir / "throughput_predictions.csv"
    pd.DataFrame(rows).to_csv(pred_csv, index=False, encoding="utf-8-sig")
    summary = {"pair_name": pair, "yolo_alias": yolo_alias, "cnn_alias": cnn_alias, "yolo_ckpt": str(yolo_ckpt), "cnn_ckpt": str(cnn_ckpt), "num_images": len(rows), "accuracy": correct/len(rows), "elapsed_seconds": elapsed, "images_per_second": len(rows)/elapsed, "milliseconds_per_image": elapsed/len(rows)*1000, "median_ms": statistics.median(ms), "p90_ms": statistics.quantiles(ms, n=10)[8] if len(ms) >= 10 else max(ms), "p95_ms": statistics.quantiles(ms, n=20)[18] if len(ms) >= 20 else max(ms), "min_ms": min(ms), "max_ms": max(ms), "model_load_seconds": load_s, "total_boxes": int(sum(row["num_boxes"] for row in rows)), "total_valid_crops": int(sum(row["num_valid_crops"] for row in rows)), "total_conf_retries": int(sum(row["conf_retries"] for row in rows)), "batch_crops": BATCH_CROPS}
    print(f"Saved predictions: {pred_csv}")
    print(f"Images/s={summary['images_per_second']:.3f} | ms/img={summary['milliseconds_per_image']:.3f} | acc={summary['accuracy']:.4f}")
    del yolo_model, cnn_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

In [ ]:
# CELL 7: chạy benchmark và lưu summary
summaries = []
for yolo_alias, yolo_ckpt in available_yolo.items():
    for cnn_alias, cnn_ckpt in available_cnn.items():
        summaries.append(benchmark_pair(yolo_alias, yolo_ckpt, cnn_alias, cnn_ckpt))

summary_df = pd.DataFrame(summaries).sort_values("images_per_second", ascending=False).reset_index(drop=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_csv = BENCHMARK_ROOT / f"model_throughput_{stamp}.csv"
summary_json = BENCHMARK_ROOT / f"model_throughput_{stamp}.json"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2, ensure_ascii=False)
print("Saved summary CSV :", summary_csv)
print("Saved summary JSON:", summary_json)
display(summary_df[["pair_name", "num_images", "images_per_second", "milliseconds_per_image", "median_ms", "p95_ms", "model_load_seconds", "accuracy", "total_boxes", "total_valid_crops", "total_conf_retries"]])


BENCHMARK: YOLOv11m__EfficientNetB2
YOLO: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/YOLOv11/YOLOv11m/weights/best.pt
CNN : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/CNN/EfficientNetB2/best_model.keras


YOLOv11m__EfficientNetB2:   0%|          | 0/291 [00:00<?, ?it/s]

Saved predictions: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/YOLOv11m__EfficientNetB2/throughput_predictions.csv
Images/s=1.027 | ms/img=973.477 | acc=0.9485

BENCHMARK: YOLOv11m__DenseNet121
YOLO: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/YOLOv11/YOLOv11m/weights/best.pt
CNN : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/CNN_Baselines/DenseNet121/best_model.keras


YOLOv11m__DenseNet121:   0%|          | 0/291 [00:00<?, ?it/s]

Saved predictions: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/YOLOv11m__DenseNet121/throughput_predictions.csv
Images/s=1.957 | ms/img=510.956 | acc=0.9416

BENCHMARK: YOLOv11m__ResNet50
YOLO: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/YOLOv11/YOLOv11m/weights/best.pt
CNN : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/CNN_Baselines/ResNet50/best_stage2.keras


YOLOv11m__ResNet50:   0%|          | 0/291 [00:00<?, ?it/s]

Saved predictions: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/YOLOv11m__ResNet50/throughput_predictions.csv
Images/s=2.103 | ms/img=475.402 | acc=0.9450

BENCHMARK: YOLOv11m__VGG16
YOLO: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/YOLOv11/YOLOv11m/weights/best.pt
CNN : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/CNN_Baselines/VGG16/best_model.keras


YOLOv11m__VGG16:   0%|          | 0/291 [00:00<?, ?it/s]

Saved predictions: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/YOLOv11m__VGG16/throughput_predictions.csv
Images/s=2.127 | ms/img=470.091 | acc=0.9347
Saved summary CSV : /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/model_throughput_20260703_155451.csv
Saved summary JSON: /content/drive/MyDrive/Papaya_Leaf_Disease_Classification/Model/Performance_Benchmark/model_throughput_20260703_155451.json


,pair_name,num_images,images_per_second,milliseconds_per_image,median_ms,p95_ms,model_load_seconds,accuracy,total_boxes,total_valid_crops,total_conf_retries
0,YOLOv11m__VGG16,291,2.127249,470.090766,197.320900,1561.232771,10.259137,0.934708,1347,1339,10
1,YOLOv11m__ResNet50,291,2.103483,475.401998,194.441856,1606.963494,14.188232,0.945017,1347,1339,10
2,YOLOv11m__DenseNet121,291,1.957117,510.955576,212.420662,1984.419586,8.556214,0.941581,1347,1339,10
3,YOLOv11m__EfficientNetB2,291,1.027245,973.477142,683.575565,2434.796852,19.051403,0.948454,1347,1339,10
